# Flight Delay Risk · Spark MLlib

# Flight Delay Risk · 2025 US domestic flights

**Тема:** предсказание задержки прибытия не менее 15 минут (`ArrDel15`) по расписанию и маршруту рейса.

- **Реальные данные:** Bureau of Transportation Statistics (U.S. DOT), таблица Reporting Carrier On-Time Performance, январь–декабрь 2025 года.
- **Масштаб:** годовые месячные выгрузки — миллионы наблюдений; число валидных рейсов будет рассчитано фактически после загрузки и фильтрации.
- **Методы:** Spark MLlib Logistic Regression на полной train-выборке и Random Forest на ограниченной случайной подвыборке train (для контроля времени/памяти). Оба обучаются распределённо, а scoring/confusion matrix считаются через Spark.
- **Защита от leakage:** не используем фактическую задержку, время фактического прибытия/отправления, длительность фактического рейса, отмену/перенаправление и сам target как признаки.
- **Лимиты:** Spark driver `4g`, executor `8g`, суммарно до `5` Spark cores; raw ZIP/CSV обрабатываются по одному месяцу и затем удаляются.

**Источник и описание полей:** [BTS TranStats / PREZIP](https://transtats.bts.gov/PREZIP/), [каталог данных U.S. DOT](https://catalog.data.gov/dataset/bts-flight-data).

> Важно: метрики здесь измеряют качество на случайном holdout одного календарного года. Для честного прогноза в будущем лучше дополнительно проверять временной split (например, первые 10 месяцев → последние 2 месяца).

> **Важно для Docker Spark:** каталог `./data` на хосте должен быть смонтирован как `/opt/spark/data` и в сервис `ml`, и в `spark-worker`. Ноутбук заранее проверяет доступ из Spark executor и останавливается с понятной инструкцией, если общий mount отсутствует. После изменения Compose пересоздайте контейнеры.


## 0. Environment and Spark session

Все тяжёлые таблицы живут в Spark. Драйвер не использует `toPandas()` и не загружает полную выборку в локальную память. Shared volume `/opt/spark/data` должен быть смонтирован одинаковым путём в `ml` и `spark-worker` контейнерах.

In [ ]:
TASK = "flight_delay"
APP_NAME = "Flight Delay Risk · Spark MLlib"

import os, json, math, shutil, zipfile, warnings
from pathlib import Path
from datetime import datetime

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, IntegerType
from pyspark import StorageLevel
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression, RandomForestClassifier
from pyspark.ml.evaluation import BinaryClassificationEvaluator

warnings.filterwarnings("ignore")
SEED = 42

def find_project_root(start=None):
    p = Path(start or Path.cwd()).resolve()
    for parent in [p, *p.parents]:
        if (parent / "requirements.txt").exists() and (parent / "notebooks").exists():
            return parent
    # Container deployments typically mount repository at /app.
    if Path("/app/requirements.txt").exists() and Path("/app/notebooks").exists():
        return Path("/app")
    raise RuntimeError("Корень проекта не найден: ожидаются requirements.txt и notebooks/.")

PROJECT = find_project_root()
DATA_ROOT = Path(os.environ.get("SPARK_DATA_ROOT", "/opt/spark/data"))
DATA_ROOT.mkdir(parents=True, exist_ok=True)
MODEL_DIR = PROJECT / "models" / "classification" / TASK
ART_DIR = PROJECT / "artifacts" / "classification" / TASK
MODEL_DIR.mkdir(parents=True, exist_ok=True)
ART_DIR.mkdir(parents=True, exist_ok=True)

master_url = os.environ.get("SPARK_MASTER_URL", "local[2]")
builder = (SparkSession.builder.appName(APP_NAME)
    .master(master_url)
    .config("spark.driver.host", os.environ.get("SPARK_DRIVER_HOST", "ml-playground"))
    .config("spark.driver.bindAddress", "0.0.0.0")
    .config("spark.driver.memory", "4g")
    .config("spark.executor.memory", "8g")
    .config("spark.executor.cores", "5")
    .config("spark.cores.max", "5")
    .config("spark.driver.maxResultSize", "512m")
    .config("spark.sql.shuffle.partitions", "20")
    .config("spark.default.parallelism", "20")
    .config("spark.sql.adaptive.enabled", "true")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "file:///opt/spark/events"))
spark = builder.getOrCreate()
spark.sparkContext.setLogLevel("WARN")

# Проверяем shared volume ДО скачивания больших данных.
# Spark executors должны видеть ровно тот же абсолютный путь, что и notebook driver.
import socket, uuid


def _shared_path_probe(paths):
    import os
    import socket
    for path in paths:
        yield (socket.gethostname(), path, os.path.exists(path))


def assert_shared_data_root():
    probe_path = DATA_ROOT / f".spark_shared_volume_probe_{uuid.uuid4().hex}.txt"
    probe_path.write_text("shared-volume-ok", encoding="utf-8")
    partitions = max(4, min(16, int(spark.sparkContext.defaultParallelism or 4)))
    try:
        checks = (
            spark.sparkContext
            .parallelize([str(probe_path)] * partitions, partitions)
            .mapPartitions(_shared_path_probe)
            .collect()
        )
    finally:
        probe_path.unlink(missing_ok=True)

    missing_hosts = sorted({host for host, path, exists in checks if not exists})
    if not checks or missing_hosts:
        details = ", ".join(missing_hosts) if missing_hosts else "проверка не вернула результатов"
        raise RuntimeError(
            "Spark executor не видит shared data volume. "
            f"Путь драйвера: {DATA_ROOT}; executor(ы) без доступа: {details}. "
            "В docker-compose.yml добавьте `./data:/opt/spark/data` в volumes сервисов `ml` И `spark-worker`, "
            "затем пересоздайте контейнеры командой `docker compose --profile spark up -d --force-recreate ml spark-master spark-worker`. "
            "После этого перезапустите kernel и выполните notebook с первой ячейки."
        )
    hosts = sorted({host for host, path, exists in checks})
    print(f"Shared volume OK: {DATA_ROOT} виден Spark executor(ам): {', '.join(hosts)}")


assert_shared_data_root()


print("PROJECT   :", PROJECT)
print("DATA_ROOT :", DATA_ROOT)
print("MODEL_DIR :", MODEL_DIR)
print("ART_DIR   :", ART_DIR)
print("SPARK     :", spark.version, "| master =", spark.sparkContext.master)


## 1. Загрузка и подготовка реального датасета

Исходные данные скачиваются с официального источника; подготовленная таблица сохраняется в Parquet в Docker shared volume. Чтобы принудительно пересобрать датасет, включите `REBUILD_DATA = True` в ячейке подготовки.

In [ ]:
TASK = "flight_delay"
APP_NAME = "ML Explorer · Flight Delay Risk · 2025"
DATASET_DIR = DATA_ROOT / TASK
RAW_DIR = DATASET_DIR / "raw"
CLEAN_DIR = DATASET_DIR / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)

# Официальный BTS PREZIP публикует отдельный ZIP на каждый месяц.
BTS_URL = "https://transtats.bts.gov/PREZIP/On_Time_Reporting_Carrier_On_Time_Performance_1987_present_2025_{month}.zip"
MONTHS = list(range(1, 13))
REBUILD_DATA = False  # True — пересобрать Parquet из исходных месячных выгрузок.
READY_MARKER = DATASET_DIR / "processed.ready"

if REBUILD_DATA:
    if CLEAN_DIR.exists(): shutil.rmtree(CLEAN_DIR)
    READY_MARKER.unlink(missing_ok=True)

if not READY_MARKER.exists():
    if CLEAN_DIR.exists(): shutil.rmtree(CLEAN_DIR)
    CLEAN_DIR.mkdir(parents=True, exist_ok=True)
    for month in MONTHS:
        zip_path = RAW_DIR / f"bts_2025_{month:02d}.zip"
        extract_dir = RAW_DIR / f"extract_{month:02d}"
        if not zip_path.exists() or not zipfile.is_zipfile(zip_path):
            url = BTS_URL.format(month=month)
            print(f"Downloading month {month:02d}/12: {url}")
            with requests.get(url, stream=True, timeout=(30, 180)) as response:
                response.raise_for_status()
                tmp_path = zip_path.with_suffix(".zip.part")
                with open(tmp_path, "wb") as f:
                    for chunk in response.iter_content(chunk_size=1024 * 1024):
                        if chunk: f.write(chunk)
                tmp_path.replace(zip_path)
        if extract_dir.exists(): shutil.rmtree(extract_dir)
        extract_dir.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(zip_path) as zf:
            csv_members = [name for name in zf.namelist() if name.lower().endswith(".csv")]
            if not csv_members:
                raise RuntimeError(f"No CSV found inside {zip_path}")
            csv_path = extract_dir / Path(csv_members[0]).name
            with zf.open(csv_members[0]) as src, open(csv_path, "wb") as dst:
                shutil.copyfileobj(src, dst, length=1024 * 1024)

        # Выборка полей и запись каждого месяца в единый shared volume.
        raw = (spark.read.option("header", "true").option("inferSchema", "false").csv(str(csv_path)))
        required = ["ArrDel15", "Cancelled", "Diverted", "Month", "DayofMonth", "DayOfWeek", "CRSDepTime", "CRSArrTime", "Distance", "CRSElapsedTime", "Reporting_Airline", "Origin", "Dest"]
        missing = [c for c in required if c not in raw.columns]
        if missing:
            raise RuntimeError(f"BTS file missing expected columns {missing}; actual columns start with {raw.columns[:30]}")
        prepared = raw.select(*required)
        numeric_casts = ["ArrDel15", "Cancelled", "Diverted", "Month", "DayofMonth", "DayOfWeek", "CRSDepTime", "CRSArrTime", "Distance", "CRSElapsedTime"]
        for c in numeric_casts:
            prepared = prepared.withColumn(c, F.col(c).cast("double"))
        prepared = (prepared
            .filter((F.col("Cancelled") == 0) & (F.col("Diverted") == 0) & F.col("ArrDel15").isin(0.0, 1.0))
            .filter(F.col("Distance").isNotNull() & (F.col("Distance") > 0))
            .filter(F.col("CRSElapsedTime").isNotNull() & (F.col("CRSElapsedTime") > 0))
            .filter(F.col("Month").isNotNull() & F.col("DayofMonth").isNotNull() & F.col("DayOfWeek").isNotNull())
            .filter(F.col("CRSDepTime").isNotNull() & F.col("CRSArrTime").isNotNull())
            .filter(F.col("Reporting_Airline").isNotNull() & F.col("Origin").isNotNull() & F.col("Dest").isNotNull())
            .select(
                F.col("ArrDel15").cast("double").alias("label"),
                F.col("Month").alias("month"), F.col("DayofMonth").alias("day_of_month"), F.col("DayOfWeek").alias("day_of_week"),
                F.col("CRSDepTime").alias("scheduled_dep_hhmm"), F.col("CRSArrTime").alias("scheduled_arr_hhmm"),
                F.col("Distance").alias("distance_miles"), F.col("CRSElapsedTime").alias("scheduled_elapsed_min"),
                F.col("Reporting_Airline").alias("airline"), F.col("Origin").alias("origin_airport"), F.col("Dest").alias("dest_airport")
            ))
        prepared.write.mode("append").parquet(str(CLEAN_DIR))
        del raw, prepared
        shutil.rmtree(extract_dir, ignore_errors=True)
        zip_path.unlink(missing_ok=True)
        print(f"Month {month:02d} processed; temporary CSV/ZIP removed.")
    READY_MARKER.write_text(datetime.now().isoformat(), encoding="utf-8")
else:
    print("Found completed processed Parquet; skip download. Set REBUILD_DATA=True to rebuild.")

print("Prepared data:", CLEAN_DIR)


## 2. EDA и контроль распределения классов

Счётчики вычисляются Spark-агрегациями; в память Python передаются только несколько чисел/классов.

## 3. Train/test split и модели Spark MLlib

Результаты сравниваются с baseline большинства. Logistic Regression обучается на всей train-части. Random Forest обучается на train-подвыборке с лимитом строк для предсказуемого времени и памяти; test для обеих моделей полный и общий.

In [ ]:
# Метрики считаются из агрегированной confusion matrix — ни одна большая выборка
# не переносится в pandas/NumPy целиком. Spark возвращает только K×K счётчики.
def metrics_from_cm(cm):
    cm = np.asarray(cm, dtype=np.int64)
    total = int(cm.sum())
    if total == 0:
        raise ValueError("Пустая тестовая выборка")
    true_support = cm.sum(axis=1).astype(float)
    pred_support = cm.sum(axis=0).astype(float)
    tp = np.diag(cm).astype(float)
    precision = np.divide(tp, pred_support, out=np.zeros_like(tp), where=pred_support > 0)
    recall = np.divide(tp, true_support, out=np.zeros_like(tp), where=true_support > 0)
    f1 = np.divide(2 * precision * recall, precision + recall, out=np.zeros_like(tp), where=(precision + recall) > 0)
    accuracy = float(tp.sum() / total)
    weighted_f1 = float(np.average(f1, weights=true_support)) if true_support.sum() else 0.0
    numerator = float(total * tp.sum() - np.dot(true_support, pred_support))
    denom_left = float(total**2 - np.dot(pred_support, pred_support))
    denom_right = float(total**2 - np.dot(true_support, true_support))
    denom = math.sqrt(max(denom_left, 0.0) * max(denom_right, 0.0))
    mcc = numerator / denom if denom > 0 else 0.0
    expected = float(np.dot(true_support, pred_support) / (total**2))
    kappa = (accuracy - expected) / (1.0 - expected) if expected < 1.0 else 0.0
    metrics = {
        "accuracy": accuracy,
        "balanced_accuracy": float(recall.mean()),
        "precision_macro": float(precision.mean()),
        "recall_macro": float(recall.mean()),
        "f1_macro": float(f1.mean()),
        "f1_weighted": weighted_f1,
        "mcc": float(mcc),
        "kappa": float(kappa),
    }
    per_class = {}
    for i, label in enumerate(CLASSES):
        per_class[label] = {
            "precision": float(precision[i]),
            "recall": float(recall[i]),
            "f1": float(f1[i]),
            "support": int(true_support[i]),
            "is_rare": label == RARE_CLASS,
        }
    return metrics, per_class

def confusion_from_predictions(predictions):
    matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int64)
    rows = predictions.groupBy(F.col("label").cast("int").alias("actual"), F.col("prediction").cast("int").alias("predicted")).count().collect()
    for row in rows:
        a, p = int(row["actual"]), int(row["predicted"])
        if 0 <= a < len(CLASSES) and 0 <= p < len(CLASSES):
            matrix[a, p] = int(row["count"])
    return matrix

def baseline_confusion(test_counts, majority_index):
    matrix = np.zeros((len(CLASSES), len(CLASSES)), dtype=np.int64)
    for actual_idx, count in enumerate(test_counts):
        matrix[actual_idx, majority_index] = int(count)
    return matrix

def json_safe(value):
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating, float)):
        val = float(value)
        return val if math.isfinite(val) else None
    if isinstance(value, (np.bool_,)):
        return bool(value)
    return value

def evaluate_model(name, model, eval_df, train_rows, model_path):
    print(f"Scoring {name} ...")
    scored = model.transform(eval_df).select("label", "prediction", "rawPrediction").persist(StorageLevel.MEMORY_AND_DISK)
    cm = confusion_from_predictions(scored)
    metrics, per_class = metrics_from_cm(cm)
    if len(CLASSES) == 2:
        for metric_name, output_name in [("areaUnderROC", "roc_auc"), ("areaUnderPR", "pr_auc")]:
            try:
                evaluator = BinaryClassificationEvaluator(labelCol="label", rawPredictionCol="rawPrediction", metricName=metric_name)
                metrics[output_name] = float(evaluator.evaluate(scored))
            except Exception as exc:
                print(f"{output_name} skipped: {exc}")
                metrics[output_name] = None
    scored.unpersist()
    return {
        "name": name,
        "is_best": False,
        "is_naive": False,
        "train_rows": int(train_rows),
        "model_path": str(model_path.relative_to(PROJECT)),
        "metrics": metrics,
        "confusion_matrix": cm.tolist(),
        "per_class": per_class,
        "per_class_recall": {label: per_class[label]["recall"] for label in CLASSES},
    }

# Простая baseline-модель: всегда выбирает самый частый класс в TRAIN.
# Её confusion matrix строится из агрегированных counts теста.
def make_baseline(train_counts, test_counts):
    majority_idx = int(np.argmax(train_counts))
    cm = baseline_confusion(test_counts, majority_idx)
    metrics, per_class = metrics_from_cm(cm)
    if len(CLASSES) == 2:
        metrics["roc_auc"] = None
        metrics["pr_auc"] = None
    return {
        "name": "Baseline · majority class",
        "is_best": False,
        "is_naive": True,
        "train_rows": 0,
        "metrics": metrics,
        "confusion_matrix": cm.tolist(),
        "per_class": per_class,
        "per_class_recall": {label: per_class[label]["recall"] for label in CLASSES},
    }

def build_feature_stages(numeric_cols, categorical_cols):
    stages = []
    indexed_cols, encoded_cols = [], []
    if categorical_cols:
        indexed_cols = [f"{c}_idx" for c in categorical_cols]
        encoded_cols = [f"{c}_ohe" for c in categorical_cols]
        stages.append(StringIndexer(inputCols=categorical_cols, outputCols=indexed_cols, handleInvalid="keep"))
        stages.append(OneHotEncoder(inputCols=indexed_cols, outputCols=encoded_cols, handleInvalid="keep", dropLast=False))
    assembler_inputs = list(numeric_cols) + encoded_cols
    stages.append(VectorAssembler(inputCols=assembler_inputs, outputCol="features", handleInvalid="keep"))
    return stages

def create_pipeline(classifier, numeric_cols, categorical_cols):
    return Pipeline(stages=build_feature_stages(numeric_cols, categorical_cols) + [classifier])

# В выборках несколько миллионов строк, поэтому не используем toPandas() по raw data.


In [ ]:
TASK = "flight_delay"  # defined again to keep this cell independently readable
APP_NAME = "ML Explorer · Flight Delay Risk · 2025"
CLASSES = ["On time / delay <15 min", "Delayed ≥15 min"]
RARE_CLASS = None
SOURCE = "U.S. DOT · Bureau of Transportation Statistics (BTS)"
SOURCE_URL = "https://transtats.bts.gov/PREZIP/"
TITLE = "Flight Delay Risk · US flights"
SUBTITLE = "Риск задержки прибытия на 15+ минут · все валидные рейсы за 2025 год"
DESCRIPTION = "Признаки известны до вылета: календарь, плановое время, авиакомпания и пара аэропортов. Реальная задержка/фактические времена не используются как признаки."
NUMERIC_COLS = ["month", "day_of_month", "day_of_week", "scheduled_dep_hhmm", "scheduled_arr_hhmm", "distance_miles", "scheduled_elapsed_min"]
CATEGORICAL_COLS = ["airline", "origin_airport", "dest_airport"]
RF_TRAIN_CAP = 400_000

# Валидируем только небольшой набор статистик; основная таблица остаётся в Spark.
df = spark.read.parquet(str(CLEAN_DIR)).persist(StorageLevel.MEMORY_AND_DISK)
n_samples = df.count()
if n_samples < 100_000:
    raise RuntimeError(f"В датасете подозрительно мало строк: {n_samples:,}. Проверьте загрузку BTS.")
counts_all_rows = df.groupBy("label").count().collect()
class_counts_idx = [0, 0]
for row in counts_all_rows: class_counts_idx[int(float(row["label"]))] = int(row["count"])
RARE_CLASS = CLASSES[int(np.argmin(class_counts_idx))]

print(f"Rows: {n_samples:,} | features: {len(NUMERIC_COLS) + len(CATEGORICAL_COLS)}")
for label, count in zip(CLASSES, class_counts_idx): print(f"  {label}: {count:,} ({count/n_samples:.2%})")
fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(CLASSES, class_counts_idx, color=["#6366f1", "#fb7185"])
ax.set_title("BTS 2025 · class distribution")
ax.set_ylabel("Flights")
ax.tick_params(axis="x", rotation=8)
plt.tight_layout(); plt.show()

train, test = df.randomSplit([0.80, 0.20], seed=SEED)
train = train.persist(StorageLevel.MEMORY_AND_DISK)
test = test.persist(StorageLevel.MEMORY_AND_DISK)
n_train, n_test = train.count(), test.count()
df.unpersist()
if n_train == 0 or n_test == 0: raise RuntimeError("Train/test split пуст")
train_counts_rows = train.groupBy("label").count().collect()
test_counts_rows = test.groupBy("label").count().collect()
train_counts = [0, 0]; test_counts = [0, 0]
for row in train_counts_rows: train_counts[int(float(row["label"]))] = int(row["count"])
for row in test_counts_rows: test_counts[int(float(row["label"]))] = int(row["count"])

results = [make_baseline(train_counts, test_counts)]
fitted_models = {}

lr = LogisticRegression(labelCol="label", featuresCol="features", maxIter=25, regParam=0.02, elasticNetParam=0.0, standardization=True)
lr_pipeline = create_pipeline(lr, NUMERIC_COLS, CATEGORICAL_COLS)
lr_path = MODEL_DIR / "flight_delay_logistic_regression_spark"
print("Fit Logistic Regression on the complete train split ...")
lr_model = lr_pipeline.fit(train)
lr_model.write().overwrite().save(str(lr_path))
fitted_models["Logistic Regression · full train"] = lr_model
results.append(evaluate_model("Logistic Regression · full train", lr_model, test, n_train, lr_path))

rf_fraction = min(1.0, RF_TRAIN_CAP / max(n_train, 1))
rf_train = train.sample(withReplacement=False, fraction=rf_fraction, seed=SEED + 1).persist(StorageLevel.MEMORY_AND_DISK)
rf_rows = rf_train.count()
rf = RandomForestClassifier(labelCol="label", featuresCol="features", numTrees=18, maxDepth=9, maxBins=32, featureSubsetStrategy="sqrt", seed=SEED)
rf_pipeline = create_pipeline(rf, NUMERIC_COLS, CATEGORICAL_COLS)
rf_path = MODEL_DIR / "flight_delay_random_forest_spark"
print(f"Fit Random Forest on a Spark sample of {rf_rows:,} rows (cap≈{RF_TRAIN_CAP:,}) ...")
rf_model = rf_pipeline.fit(rf_train)
rf_model.write().overwrite().save(str(rf_path))
fitted_models["Random Forest · Spark sample"] = rf_model
results.append(evaluate_model("Random Forest · Spark sample", rf_model, test, rf_rows, rf_path))
rf_train.unpersist()

best = max([r for r in results if not r.get("is_naive")], key=lambda r: r["metrics"].get("f1_macro", -1))
for result in results: result["is_best"] = result["name"] == best["name"]

class_distribution = {CLASSES[int(float(row["label"]))]: int(row["count"]) for row in counts_all_rows}
max_count, min_count = max(class_counts_idx), max(1, min(class_counts_idx))
meta = {
    "id": TASK, "title": TITLE, "subtitle": SUBTITLE, "description": DESCRIPTION,
    "source": SOURCE, "source_url": SOURCE_URL, "generated_at": datetime.now().isoformat(),
    "hero_metric": "f1_macro", "target_name": "ArrDel15",
    "classes": CLASSES,
    "dataset_stats": {"n_samples": int(n_samples), "n_features": len(NUMERIC_COLS) + len(CATEGORICAL_COLS), "n_classes": len(CLASSES), "n_train": int(n_train), "n_test": int(n_test)},
    "target_stats": {"class_distribution": class_distribution, "rare_class": RARE_CLASS, "imbalance_ratio": float(max_count / min_count), "rare_class_ratio": float(min_count / n_samples)},
}
report = {"meta": meta, "models": results}
report_path = ART_DIR / "report.json"
report_path.write_text(json.dumps(json_safe(report), ensure_ascii=False, indent=2, allow_nan=False), encoding="utf-8")
pd.DataFrame([{ "model": r["name"], **r["metrics"], "train_rows": r.get("train_rows", 0) } for r in results]).to_csv(ART_DIR / "metrics_all_models.csv", index=False)
print("Best model by macro-F1:", best["name"], "| macro-F1 =", round(best["metrics"]["f1_macro"], 4))
print("Report:", report_path)

fig, ax = plt.subplots(figsize=(9, 4.5))
plot_rows = [r for r in results if r.get("metrics", {}).get("f1_macro") is not None]
ax.bar([r["name"] for r in plot_rows], [r["metrics"]["f1_macro"] for r in plot_rows], color=["#94a3b8" if r.get("is_naive") else ("#10b981" if r.get("is_best") else "#6366f1") for r in plot_rows])
ax.set_title("Flight Delay · macro-F1 on the test split"); ax.set_ylim(0, 1); ax.tick_params(axis="x", rotation=15)
plt.tight_layout(); plt.show()


## 4. Сохранённые артефакты

Ноутбук создаёт Spark PipelineModel в `models/classification/<task>/` и `report.json` + `metrics_all_models.csv` в `artifacts/classification/<task>/`. Фронтенд читает `report.json`; перезапустите страницу после завершения ноутбука. Для повторного запуска Spark можно остановить ячейкой ниже.

In [ ]:
print("Model directory:", MODEL_DIR)
for item in sorted(MODEL_DIR.iterdir()):
    print(" ", item.name)
print("Artifact directory:", ART_DIR)
for item in sorted(ART_DIR.iterdir()):
    print(" ", item.name, f"({item.stat().st_size / 1024:.1f} KB)" if item.is_file() else "<directory>")

# Раскомментируйте, если закончена работа в текущей Spark-сессии.
# spark.stop()